In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"
 
print(os.getcwd())
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print(os.getcwd())

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick 
import numpy as np
from model_utils.model_config import get_model_path
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [ ]:
model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b", "watt-tool-8b"]

model_dsiplay_name_dict = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "qwen3-14b": "Qwen3-14B",
    "toolace-2.5-8b": "ToolACE-2.5-8B",
    "watt-tool-8b": "Watt-Tool-8B"
}

topn_dict = {
    "qwen3-4b": [11,23,34],
    "qwen3-8b": [11,23,34],
    "qwen3-14b": [16,32,48],
    "toolace-2.5-8b": [10, 20, 30],
    "watt-tool-8b": [10, 20, 30]
}

file_name_general = "num_500_add_1_trunc_500_test_ap_logit_diff_fix_topn_{topn}_results.json"

In [ ]:
result = {}
for model_name in model_name_list:
    topn_list = topn_dict[model_name]

    result[model_dsiplay_name_dict[model_name]] = {}

    for topn in topn_list:
        file_name = file_name_general.format(topn=topn)

        result[model_dsiplay_name_dict[model_name]][topn] = {}

        file_path = os.path.join("results", model_name, "scale_search_single_test", file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            result[model_dsiplay_name_dict[model_name]][topn] = json.load(f)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.patheffects as path_effects

def plot_heatmap_strips_style_fix(model_data_topn, model_name, topn_value):
    
    data_list = []
    target_metric = "tool_calling_rate"
    
    
    for key, val in model_data_topn.items():
        parts = key.split('_')
        if len(parts) != 3: continue
        comp_type = parts[0] # sem / str
        # if int(parts[2][-1])%2:
        
        #     continue
        try:
            scale_val = float(parts[2])
        except ValueError: continue
        metric_val = val["pair_data"][target_metric]
        
        data_list.append({
            'Type': 'Semantic' if comp_type == 'sem' else 'Structural',
            'Scale': scale_val,
            'Value': metric_val
        })
    
    df = pd.DataFrame(data_list)
    df['Scale'] = df['Scale'].round(1)
    df_pivot = df.pivot(index='Type', columns='Scale', values='Value')
    df_pivot = df_pivot.reindex(['Semantic', 'Structural'])

    # Baseline (rho=1.0) as heatmap centre
    baseline_val = 0.5 
    if 1.0 in df_pivot.columns:
        
        baseline_val = df_pivot[1.0].mean()
    else:
        
        baseline_val = df_pivot.values.mean()

    # Plot config
    plt.rcParams['text.usetex'] = False 
    plt.rcParams['mathtext.fontset'] = 'cm' 
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica'] 
    
    
    sns.set_theme(style="ticks", font_scale=1.6, rc={"font.family": "sans-serif"})
    
    fig, ax = plt.subplots(figsize=(12, 3.5)) 
    
    # Heatmap
    
    
    sns.heatmap(df_pivot, ax=ax, cmap='RdBu_r',
                center=baseline_val,
                annot=True, fmt=".2f", 
                annot_kws={"size": 12},
                linewidths=0, linecolor='black', 
                cbar_kws={'label': "Tool Invocation Rate", 'pad': 0.02})
                
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1) 

    cbar = ax.collections[0].colorbar
    
    cbar.outline.set_visible(True)
    cbar.outline.set_edgecolor('black')
    cbar.outline.set_linewidth(1) 


    
    ax.set_title(f'Model: {model_name} (Top-'+r'$k$'+f': {topn_value})', 
                 fontsize=19, fontweight='bold', pad=14)
    
    
    ax.set_xlabel('Scaling Coefficient', fontsize=19, fontweight='bold')
    ax.set_ylabel('') 
    
    
    plt.xticks(rotation=0, fontsize=14)
    plt.yticks(rotation=90, fontweight='bold', fontsize=16)
    
    
    # scales = df_pivot.columns.tolist()
    # if 1.0 in scales:
    #     idx_1 = scales.index(1.0)
    
    #     rect = plt.Rectangle((idx_1, 0), 1, 2, fill=False, edgecolor='black', lw=1, clip_on=False)
    #     ax.add_patch(rect)
    
    #     # ax.text(idx_1 + 0.5, 2.1, 'Baseline', ha='center', va='top', 
    #     #         color='black', fontsize=12, fontweight='bold')

    plt.tight_layout()
    
    safe_name = f"{model_name}".replace(" ", "_")
    plt.savefig(f'figs/mech/scale_single/top_{topn_value}_{safe_name}_app.pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:

for model_name in model_name_list:
    model_key = model_dsiplay_name_dict[model_name]
    for topn_val in topn_dict[model_name]:
        plot_heatmap_strips_style_fix(result[model_key][topn_val], model_key, topn_val)